In [1]:
import TwoPUtils
%load_ext autoreload
%autoreload 2
from reward_relative.path_dict_seahorse import path_dictionary as path_dict
from reward_relative import utilities as ut
from reward_relative import plotUtils as pt
from reward_relative import spatial
from reward_relative import placeCellPlot
from reward_relative import dayData as dd
from reward_relative import behavior
import pickle
import dill
import numpy as np
import os 
import matplotlib.pyplot as plt
import TwoPUtils
import protter_functions as pf
from scipy.signal import medfilt
from matplotlib.lines import Line2D
import pandas as pd

import importlib
import h5py
import traceback
import sys
sys.path.append('/home/sosalab/local_repos/InVivoDA_analyses') 

In [2]:

experiment = 'MetaLearn'
year = 'combined'
exp_days = [3, 5, 7, 8, 10, 12, 14]

max_anim_list = dd.max_anim_list(experiment, exp_days, year=year)

## These parameters were used for computing the saved multiDayData
# bin_size = 10  # for quantifying distribution of place field peak locations
# sigma = 1  # for smoothing
# smooth = False  # whether to smooth for finding place cell peaks
# exclude_int = True  # exclude putative interneurons
# int_thresh = 0.5

## Place cell logical definitions:
## 'and' = must have significant spatial information
## in trial set 0 AND trial set 1 (i.e. before and after the reward switch)
## 'or' = must have signitive spatial information in trial set 0 OR trial set 1

# place_cell_logical = 'or'
ts_key = 'dff'  # which timeseries to use for finding peaks
# use_speed_thr = True  # use a speed threshold to calculate new trial matrices
# # speed threshold in cm/s (excludes data at speed less than this)
# speed_thr = 2

reward_dist_inclusive = 50  # in cm

# datetime of saved file
dt = "202504"

pkl_name = "%s_expdays%s_multiDayData_%s_%s.pickle" % (
    # ut.make_anim_tag(max_anim_list),
    f'm{ut.get_mouse_number(max_anim_list[0])}-{ut.get_mouse_number(max_anim_list[-1])}',
    ut.make_day_tag(
        exp_days),
    ts_key,
    dt)
pkl_path = os.path.join(
    path_dict['preprocessed_root'], 'multi_anim_sess', pkl_name)
print(pkl_path)
multiDayData = dill.load(open(pkl_path, "rb"), ignore=True)

include_ans = multiDayData[exp_days[-1]
                           ].circ_rel_stats_across_an['include_ans']
max_anim_list = sorted(np.unique(np.concatenate([multiDayData[day].anim_list
                                                 for day in exp_days])),
                       key=len)
include_ans

/data/2p_data/multi_anim_sess/m2-19_expdays3-5-7-8-10-12-14_multiDayData_dff_202504.pickle


array(['GCAMP3', 'GCAMP4', 'GCAMP7', 'GCAMP11', 'GCAMP12', 'GCAMP13',
       'GCAMP14', 'GCAMP15', 'GCAMP17', 'GCAMP18', 'GCAMP19'], dtype='<U7')

In [3]:
pkl_path = '/data/2p_data/multi_anim_sess/m2-19_expdays1-2-3-4-5-6-7-8-9-10-11-12-13-14_multiDayData_dff_202504.pickle'

multiDayData = dill.load(open(pkl_path, "rb"), ignore=True)

In [4]:
animal_lists_by_day = {d:multiDayData[d].anim_list for d in multiDayData.keys()}

In [5]:
animal_lists_by_day.keys()

dict_keys([1, 2, 4, 6, 9, 11, 13, 3, 5, 7, 8, 10, 12, 14])

In [6]:
all_exp_days = [i for i in range(3, 15, 1)]

In [7]:
all_animals = []
for _, anim_list in animal_lists_by_day.items():
    all_animals += list(anim_list)

all_animals = set(all_animals)
print(all_animals)

{'GCAMP3', 'GCAMP5', 'GCAMP15', 'GCAMP11', 'GCAMP13', 'GCAMP4', 'GCAMP12', 'GCAMP19', 'GCAMP18', 'GCAMP6', 'GCAMP2', 'GCAMP14', 'GCAMP17', 'GCAMP10', 'GCAMP7'}


In [16]:
ma_ses = dd.load_multi_anim_sess(path_dict, 5, [ani for ani in animal_lists_by_day[5] if ani != 'GCAMP5'], subdirs = ['multi_anim_sess'])

In [19]:
test_sess = ma_ses['GCAMP14']['sess']

,time,morph,dreamland,trialnum,pos,dz,posx,lick,reward,tstart,teleport,rzone,scanning,manrewards,autoreward,cmd,speed,lick rate
0,0.000000,-1.0,-1.0,-1.0,-500.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,-1.0,0.0,0.0,-1.0,0.0,0.0
1,0.064484,-1.0,-1.0,-1.0,-500.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,-1.0,0.0,0.0,-1.0,0.0,0.0
2,0.128967,-1.0,-1.0,-1.0,-500.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,-1.0,0.0,0.0,-1.0,0.0,0.0
3,0.193451,-1.0,-1.0,-1.0,-500.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,-1.0,0.0,0.0,-1.0,0.0,0.0
4,0.257935,-1.0,-1.0,-1.0,-500.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,-1.0,0.0,0.0,-1.0,0.0,0.0


In [25]:
file_name = '/data/dave/sosa_data_h5/all_data'


if os.path.exists(file_name):
    response = input(f'file already exists! want to overwrite? (y)')
    if not response.lower() == 'y':
        raise FileExistsError('not going to overwrite your db, sorry!')




with h5py.File(file_name, "w") as f:

    animal_groups = {animal: f.create_group(animal) for animal in all_animals}


    # for i, day in enumerate(sorted(exp_days)):
    for i, day in enumerate(all_exp_days):
        print(day)

        animal_list = [ani for ani in animal_lists_by_day[day] if ani != 'GCAMP5']
        

        # load multi_anim_sess that has the spatial activity to plot
        multi_anim_sess = dd.load_multi_anim_sess(path_dict, day, animal_list,
                                            subdirs = ['multi_anim_sess'])


        
        
        #some lazy error handling here so I can go home and run this 
        try:
            for animal in animal_list:
                session = multi_anim_sess[animal]['sess']


                today = animal_groups[animal].create_group(str(day))
                today.attrs['date'] = session.date


                meta = today.create_group('trial_metadata')

                trial_info = pf.TrialInfo({day:session}).lookup_df
                licks = pf.correct_licks(session)
                licks_smoothed = pf.smooth_trial_matrix(licks)

                data, occupancy, bin_edges, bin_centers = licks

                lick_g = today.create_group('lick_data')

                lick_g.create_dataset('corrected_licks', data = data)
                lick_g.create_dataset('bin_center', data = bin_centers)
                lick_g.create_dataset('smoothed_licks', data = licks_smoothed)
                lick_g.create_dataset('occupancy', data = occupancy)
                lick_g.create_dataset('bin_edges', data = bin_edges)
                


                for key in ['speed', 'rewards']:
                    try:
                        '''unpack trial matrices and put them in their own group. i think this will make code more readable.'''
                        today_k = today.create_group(key)
                        data, occupancy, bin_edges, bin_centers = session.trial_matrices[key]

                        today_k.create_dataset(key, data = data)
                        today_k.create_dataset('bin_centers', data = bin_centers)
                        today_k.create_dataset('occupancy', data = occupancy)
                        today_k.create_dataset('bin_edges', data = bin_edges)
                    except:
                        print(f'{key} : {animal} : {day} ')
                
                for col in trial_info.columns:
                    data = trial_info[col].values

                    meta.create_dataset(col, data = data)

                vr_dat = today.create_group('vr_data')

                for col in session.vr_data.columns:
                    vr_dat.create_dataset(col, data = session.vr_data[col])

                bin_size = 5
                speed_thresh = 2
                impute_nans = True
                
                neural_tm = TwoPUtils.spatial_analyses.trial_matrix(session.timeseries['events'].T,
                                                    session.vr_data['pos']._values,
                                                    session.trial_start_inds,
                                                    session.teleport_inds,
                                                    bin_size=bin_size,
                                                    min_pos=0,
                                                    max_pos=450,
                                                    speed_thr=speed_thresh,
                                                    speed=session.vr_data['speed']._values,
                                                    impute_nans = impute_nans,
                                                    
                                                    )
    

                neural_g = today.create_group('neural_events')
                data, occupancy, bin_edges, bin_centers = neural_tm

                neural_g.create_dataset('neural_events', data = data)
                neural_g.create_dataset('bin_centers', data = bin_centers)
                neural_g.create_dataset('occupancy', data = occupancy)
                neural_g.create_dataset('bin_edges', data = bin_edges)


                
                neural_g.attrs['bin_size'] = bin_size
                neural_g.attrs['speed_thres'] = speed_thresh
                neural_g.attrs['impute_nans']= impute_nans

                time_series = today.create_group('timeseries')
                for ts_key in session.timeseries.keys():
                    time_series.create_dataset(ts_key, data =session.timeseries[ts_key] )
                time_series.create_dataset('trial_start_inds', data = session.trial_start_inds)
                time_series.create_dataset('teleport_inds', data = session.trial_start_inds)


                

                    
                    

                
        except:
            traceback.print_exc()
            raise
            
        del(multi_anim_sess)
        del(session)

3


/home/sosalab/local_repos/TwoPUtils/TwoPUtils/spatial_analyses.py:73: RuntimeWarning: Mean of empty slice
  trial_mat[trial, b] = np.nanmean(arr_t[(pos_t > edge1) & (pos_t <= edge2), :], axis=0)
/home/sosalab/local_repos/TwoPUtils/TwoPUtils/spatial_analyses.py:73: RuntimeWarning: Mean of empty slice
  trial_mat[trial, b] = np.nanmean(arr_t[(pos_t > edge1) & (pos_t <= edge2), :], axis=0)
/home/sosalab/local_repos/TwoPUtils/TwoPUtils/spatial_analyses.py:73: RuntimeWarning: Mean of empty slice
  trial_mat[trial, b] = np.nanmean(arr_t[(pos_t > edge1) & (pos_t <= edge2), :], axis=0)
/home/sosalab/local_repos/TwoPUtils/TwoPUtils/spatial_analyses.py:73: RuntimeWarning: Mean of empty slice
  trial_mat[trial, b] = np.nanmean(arr_t[(pos_t > edge1) & (pos_t <= edge2), :], axis=0)
/home/sosalab/local_repos/TwoPUtils/TwoPUtils/spatial_analyses.py:73: RuntimeWarning: Mean of empty slice
  trial_mat[trial, b] = np.nanmean(arr_t[(pos_t > edge1) & (pos_t <= edge2), :], axis=0)
/home/sosalab/local_repos

4


/home/sosalab/local_repos/TwoPUtils/TwoPUtils/spatial_analyses.py:73: RuntimeWarning: Mean of empty slice
  trial_mat[trial, b] = np.nanmean(arr_t[(pos_t > edge1) & (pos_t <= edge2), :], axis=0)
/home/sosalab/local_repos/TwoPUtils/TwoPUtils/spatial_analyses.py:73: RuntimeWarning: Mean of empty slice
  trial_mat[trial, b] = np.nanmean(arr_t[(pos_t > edge1) & (pos_t <= edge2), :], axis=0)
/home/sosalab/local_repos/TwoPUtils/TwoPUtils/spatial_analyses.py:73: RuntimeWarning: Mean of empty slice
  trial_mat[trial, b] = np.nanmean(arr_t[(pos_t > edge1) & (pos_t <= edge2), :], axis=0)
/home/sosalab/local_repos/TwoPUtils/TwoPUtils/spatial_analyses.py:73: RuntimeWarning: Mean of empty slice
  trial_mat[trial, b] = np.nanmean(arr_t[(pos_t > edge1) & (pos_t <= edge2), :], axis=0)
/home/sosalab/local_repos/TwoPUtils/TwoPUtils/spatial_analyses.py:73: RuntimeWarning: Mean of empty slice
  trial_mat[trial, b] = np.nanmean(arr_t[(pos_t > edge1) & (pos_t <= edge2), :], axis=0)
/home/sosalab/local_repos

5


...

/home/sosalab/local_repos/TwoPUtils/TwoPUtils/spatial_analyses.py:73: RuntimeWarning: Mean of empty slice
  trial_mat[trial, b] = np.nanmean(arr_t[(pos_t > edge1) & (pos_t <= edge2), :], axis=0)
/home/sosalab/local_repos/TwoPUtils/TwoPUtils/spatial_analyses.py:73: RuntimeWarning: Mean of empty slice
  trial_mat[trial, b] = np.nanmean(arr_t[(pos_t > edge1) & (pos_t <= edge2), :], axis=0)
/home/sosalab/local_repos/TwoPUtils/TwoPUtils/spatial_analyses.py:73: RuntimeWarning: Mean of empty slice
  trial_mat[trial, b] = np.nanmean(arr_t[(pos_t > edge1) & (pos_t <= edge2), :], axis=0)
/home/sosalab/local_repos/TwoPUtils/TwoPUtils/spatial_analyses.py:73: RuntimeWarning: Mean of empty slice
  trial_mat[trial, b] = np.nanmean(arr_t[(pos_t > edge1) & (pos_t <= edge2), :], axis=0)
/home/sosalab/local_repos/TwoPUtils/TwoPUtils/spatial_analyses.py:73: RuntimeWarning: Mean of empty slice
  trial_mat[trial, b] = np.nanmean(arr_t[(pos_t > edge1) & (pos_t <= edge2), :], axis=0)
/home/sosalab/local_repos

13


/home/sosalab/local_repos/TwoPUtils/TwoPUtils/spatial_analyses.py:73: RuntimeWarning: Mean of empty slice
  trial_mat[trial, b] = np.nanmean(arr_t[(pos_t > edge1) & (pos_t <= edge2), :], axis=0)
/home/sosalab/local_repos/TwoPUtils/TwoPUtils/spatial_analyses.py:73: RuntimeWarning: Mean of empty slice
  trial_mat[trial, b] = np.nanmean(arr_t[(pos_t > edge1) & (pos_t <= edge2), :], axis=0)
/home/sosalab/local_repos/TwoPUtils/TwoPUtils/spatial_analyses.py:73: RuntimeWarning: Mean of empty slice
  trial_mat[trial, b] = np.nanmean(arr_t[(pos_t > edge1) & (pos_t <= edge2), :], axis=0)
/home/sosalab/local_repos/TwoPUtils/TwoPUtils/spatial_analyses.py:73: RuntimeWarning: Mean of empty slice
  trial_mat[trial, b] = np.nanmean(arr_t[(pos_t > edge1) & (pos_t <= edge2), :], axis=0)
/home/sosalab/local_repos/TwoPUtils/TwoPUtils/spatial_analyses.py:73: RuntimeWarning: Mean of empty slice
  trial_mat[trial, b] = np.nanmean(arr_t[(pos_t > edge1) & (pos_t <= edge2), :], axis=0)
/home/sosalab/local_repos

14


/home/sosalab/local_repos/TwoPUtils/TwoPUtils/spatial_analyses.py:73: RuntimeWarning: Mean of empty slice
  trial_mat[trial, b] = np.nanmean(arr_t[(pos_t > edge1) & (pos_t <= edge2), :], axis=0)
/home/sosalab/local_repos/TwoPUtils/TwoPUtils/spatial_analyses.py:73: RuntimeWarning: Mean of empty slice
  trial_mat[trial, b] = np.nanmean(arr_t[(pos_t > edge1) & (pos_t <= edge2), :], axis=0)
/home/sosalab/local_repos/TwoPUtils/TwoPUtils/spatial_analyses.py:73: RuntimeWarning: Mean of empty slice
  trial_mat[trial, b] = np.nanmean(arr_t[(pos_t > edge1) & (pos_t <= edge2), :], axis=0)
/home/sosalab/local_repos/TwoPUtils/TwoPUtils/spatial_analyses.py:73: RuntimeWarning: Mean of empty slice
  trial_mat[trial, b] = np.nanmean(arr_t[(pos_t > edge1) & (pos_t <= edge2), :], axis=0)
/home/sosalab/local_repos/TwoPUtils/TwoPUtils/spatial_analyses.py:73: RuntimeWarning: Mean of empty slice
  trial_mat[trial, b] = np.nanmean(arr_t[(pos_t > edge1) & (pos_t <= edge2), :], axis=0)
/home/sosalab/local_repos

In [26]:
f.close()